# 05. Batch Inference — GreenSpace CNN

Run the trained model on unseen images and export predictions to CSV.

**Prerequisites:**
1. Cache images locally first:
   ```bash
   python scripts/cache_drive_folder.py --folder-id 1C5pkrcuuciTZyLPwhQqpvPZwWJ7fMBzI
   ```
2. A trained model under `models/runs/<RUN_TAG>/`

**Outputs:**
- `predictions/predictions_<RUN_TAG>.csv` — one row per image, all head outputs as columns
- Summary statistics and comparison to training distribution

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path('../').resolve()))
from src.label_schema import resolve_label_cols

# --- Auto-resolve model (same pattern as 04_model_evaluation) ---
runs_root = Path('../models/runs')
RUN_TAG = globals().get('RUN_TAG', None)

if RUN_TAG is None:
    assert runs_root.exists(), f'Missing runs directory: {runs_root}'
    run_dirs = sorted(
        [p for p in runs_root.iterdir() if p.is_dir()],
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    assert run_dirs, f'No run folders found in {runs_root}'
    run_dir = run_dirs[0]
    RUN_TAG = run_dir.name
else:
    run_dir = runs_root / RUN_TAG

# Prefer best checkpoint, fall back to final
candidates = sorted(run_dir.glob('best_*.keras')) + sorted(run_dir.glob('final_*.keras'))
assert candidates, f'No .keras model found in {run_dir}'
model_path = candidates[0]

model = tf.keras.models.load_model(model_path, compile=False)
print('RUN_TAG:', RUN_TAG)
print('Loaded model:', model_path.name)

# --- Locate cached inference images ---
INFERENCE_CACHE = Path('../data/cache/inference_images')
assert INFERENCE_CACHE.exists(), (
    f'Missing {INFERENCE_CACHE}. Run:\n'
    '  python scripts/cache_drive_folder.py --folder-id 1C5pkrcuuciTZyLPwhQqpvPZwWJ7fMBzI'
)

image_paths = sorted(INFERENCE_CACHE.glob('*.jpg')) + sorted(INFERENCE_CACHE.glob('*.jpeg')) + sorted(INFERENCE_CACHE.glob('*.png'))
print(f'Found {len(image_paths)} cached images')

# --- Load training split for comparison later (Option 5) ---
train_csv = Path('../data/processed/splits/train.csv')
assert train_csv.exists(), 'Missing train.csv — run notebook 02 first'
train_df = pd.read_csv(train_csv)

_schema = resolve_label_cols(train_df)
binary_cols   = _schema['binary_cols']
bin_names     = _schema['bin_names']

In [ ]:
# Batch inference — decode images, predict in batches, collect results.
IMG_SIZE = (512, 512)
BATCH_SIZE = 16

def decode_and_resize(path):
    img = tf.io.read_file(str(path))
    img = tf.io.decode_jpeg(img, channels=3)
    img = tf.cast(img, tf.float32) / 255.0
    return img

# Build tf.data pipeline from cached paths
path_strings = [str(p) for p in image_paths]
ds = (
    tf.data.Dataset.from_tensor_slices(path_strings)
    .map(lambda p: tf.py_function(decode_and_resize, [p], tf.float32), num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

print(f'Running inference on {len(image_paths)} images (batch_size={BATCH_SIZE})...')
preds = model.predict(ds, verbose=1)

# preds is a list of 4 arrays: [bin_head, shade_head, score_head, veg_head]
pred_bin   = preds[0]  # (N, 8) sigmoid probabilities
pred_shade = preds[1]  # (N, 2) softmax
pred_score = preds[2]  # (N, 5) softmax
pred_veg   = preds[3]  # (N, 5) softmax

print(f'Predictions complete: {pred_bin.shape[0]} images')

# Assemble results into a DataFrame
filenames = [p.name for p in image_paths]

rows = {
    'image_filename': filenames,
}

# Binary probabilities
for i, name in enumerate(bin_names):
    rows[f'{name}_prob'] = pred_bin[:, i]

# Shade: class + confidence
rows['shade_class'] = np.where(pred_shade.argmax(axis=1) == 0, 'minimal', 'abundant')
rows['shade_confidence'] = pred_shade.max(axis=1)

# Score: expected value (1–5 scale)
score_values = np.arange(1, 6, dtype=np.float32)
rows['score_ev'] = pred_score @ score_values

# Veg: expected value (1–5 scale)
rows['veg_ev'] = pred_veg @ score_values

pred_df = pd.DataFrame(rows)

# Save CSV
out_dir = Path('../predictions')
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / f'predictions_{RUN_TAG}.csv'
pred_df.to_csv(out_path, index=False)
print(f'Saved {len(pred_df)} predictions to {out_path}')
pred_df.head()

In [ ]:
# Option 1: Summary statistics across all unseen images

prob_cols = [f'{n}_prob' for n in bin_names]
THRESHOLD = 0.5

# Binary feature prevalence (% of images above threshold)
prevalence = (pred_df[prob_cols] >= THRESHOLD).mean().sort_values()
prevalence.index = [c.replace('_prob', '') for c in prevalence.index]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# 1a. Binary feature prevalence
ax = axes[0, 0]
ax.barh(prevalence.index, prevalence.values)
ax.set_xlabel(f'Fraction above {THRESHOLD}')
ax.set_title('Binary Feature Prevalence (Inference)')
ax.set_xlim(0, 1)

# 1b. Shade distribution
ax = axes[0, 1]
shade_counts = pred_df['shade_class'].value_counts()
ax.bar(shade_counts.index, shade_counts.values)
ax.set_title('Shade Distribution (Inference)')
ax.set_ylabel('Count')

# 1c. Structured score distribution
ax = axes[1, 0]
ax.hist(pred_df['score_ev'], bins=30, edgecolor='black', alpha=0.7)
ax.set_xlabel('Expected Value (1–5)')
ax.set_title('Structured–Unstructured Score (Inference)')
ax.axvline(pred_df['score_ev'].mean(), color='red', linestyle='--', label=f"mean={pred_df['score_ev'].mean():.2f}")
ax.legend()

# 1d. Vegetation distribution
ax = axes[1, 1]
ax.hist(pred_df['veg_ev'], bins=30, edgecolor='black', alpha=0.7)
ax.set_xlabel('Expected Value (1–5)')
ax.set_title('Vegetation Cover (Inference)')
ax.axvline(pred_df['veg_ev'].mean(), color='red', linestyle='--', label=f"mean={pred_df['veg_ev'].mean():.2f}")
ax.legend()

plt.tight_layout()
plt.savefig(out_dir / f'summary_stats_{RUN_TAG}.png', dpi=150, bbox_inches='tight')
plt.show()

# Print key numbers
print(f'Images: {len(pred_df)}')
print(f'\nBinary feature prevalence (>= {THRESHOLD}):')
for name, val in prevalence.items():
    print(f'  {name:<30} {val:.1%}')
print(f'\nShade: {dict(shade_counts)}')
print(f'Score EV: mean={pred_df["score_ev"].mean():.2f}, std={pred_df["score_ev"].std():.2f}')
print(f'Veg EV:   mean={pred_df["veg_ev"].mean():.2f}, std={pred_df["veg_ev"].std():.2f}')

In [ ]:
# Option 5: Compare inference predictions to training label distribution.
# Large shifts may indicate domain gap between the training set and these unseen images.

# Training: hard binary labels (0/1 from soft >= 0.5 threshold)
hard_bin_cols = [n for n in bin_names if n in train_df.columns]
train_prev = train_df[hard_bin_cols].mean().sort_values()

# Inference: predicted prevalence at same threshold
infer_prev = (pred_df[prob_cols] >= THRESHOLD).mean()
infer_prev.index = [c.replace('_prob', '') for c in infer_prev.index]
infer_prev = infer_prev.reindex(train_prev.index)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# 5a. Binary prevalence side-by-side
ax = axes[0]
x = np.arange(len(train_prev))
w = 0.35
ax.barh(x - w/2, train_prev.values, height=w, label='Training (labels)', alpha=0.8)
ax.barh(x + w/2, infer_prev.values, height=w, label='Inference (predicted)', alpha=0.8)
ax.set_yticks(x)
ax.set_yticklabels(train_prev.index)
ax.set_xlabel('Prevalence')
ax.set_title('Binary Feature: Training vs Inference')
ax.legend()
ax.set_xlim(0, 1)

# 5b. Score EV comparison
ax = axes[1]
# Training score: stored as 1-5 integer class
if 'score_class' in train_df.columns:
    train_score = train_df['score_class'].dropna()
    ax.hist(train_score, bins=np.arange(0.5, 6.5, 1), alpha=0.5, label='Training', density=True, edgecolor='black')
ax.hist(pred_df['score_ev'], bins=30, alpha=0.5, label='Inference', density=True, edgecolor='black')
ax.set_xlabel('Structured–Unstructured Score')
ax.set_title('Score Distribution: Training vs Inference')
ax.legend()

# 5c. Veg EV comparison
ax = axes[2]
if 'veg_class' in train_df.columns:
    train_veg = train_df['veg_class'].dropna()
    ax.hist(train_veg, bins=np.arange(0.5, 6.5, 1), alpha=0.5, label='Training', density=True, edgecolor='black')
ax.hist(pred_df['veg_ev'], bins=30, alpha=0.5, label='Inference', density=True, edgecolor='black')
ax.set_xlabel('Vegetation Cover Score')
ax.set_title('Vegetation Distribution: Training vs Inference')
ax.legend()

plt.tight_layout()
plt.savefig(out_dir / f'train_vs_inference_{RUN_TAG}.png', dpi=150, bbox_inches='tight')
plt.show()

# Print shift summary
print('Binary prevalence shift (inference - training):')
shift = infer_prev - train_prev
for name, val in shift.sort_values().items():
    direction = '+' if val > 0 else ''
    print(f'  {name:<30} {direction}{val:.1%}')